# n_bits vs Accuracy
Cross-validated accuracy within error thresholds.


In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

MODEL_DIR = 'random-forest'
MODEL_NAME = 'Random Forest'

cwd = Path.cwd()
project_root = Path(".." ).resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qspr_config import (
    DATA_PATH,
    ECFP_RADIUS,
    ECFP_N_BITS,
    N_BITS_GRID,
    N_ESTIMATORS,
    N_ESTIMATORS_GRID,
    RANDOM_SEED,
    CV_FOLDS,
    N_JOBS,
    FIG_DPI,
    FIGSIZE_SQUARE,
)
from qspr_common import (
    load_dataset,
    build_feature_matrix,
    apply_plot_style,
    resolve_output_dir,
)

apply_plot_style()
out_dir = resolve_output_dir(MODEL_DIR)


In [2]:
df_raw = load_dataset(DATA_PATH)


In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
thresholds = [0.3, 0.5, 1.0]

acc_means = {thr: [] for thr in thresholds}

for n_bits in N_BITS_GRID:
    df_bits, X = build_feature_matrix(df_raw, radius=ECFP_RADIUS, n_bits=n_bits)
    y = df_bits["Solubility"].to_numpy()

    accs = {thr: [] for thr in thresholds}
    for train_idx, test_idx in cv.split(X):
        model = RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            random_state=RANDOM_SEED,
            n_jobs=N_JOBS,
        )
        model.fit(X[train_idx], y[train_idx])
        pred = model.predict(X[test_idx])
        err = np.abs(pred - y[test_idx])
        for thr in thresholds:
            accs[thr].append((err <= thr).mean())

    for thr in thresholds:
        acc_means[thr].append(np.mean(accs[thr]))

fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)
for thr in thresholds:
    ax.plot(N_BITS_GRID, acc_means[thr], marker="o", label=f"|err| <= {thr}")
ax.set_xlabel("n_bits")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.0)
ax.set_title(f"{MODEL_NAME}: accuracy within threshold")
ax.set_xticks(N_BITS_GRID)
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=8)
fig.tight_layout()

out_path = out_dir / "n_bits_vs_accuracy.png"
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
out_path
